
# Actualización del monitor de producción

**Autor**: Juan Carlos Alfaro Jiménez

Esta libreta lanza la actualización del monitor de **`Databricks Lakehouse Monitoring`** configurado sobre `gold_fraud_inference_enriched` y espera a que finalice antes de continuar. Se ejecuta automáticamente como última tarea del *job* `Credit Card Fraud Feature Pipeline`, garantizando que las métricas de rendimiento y *drift* reflejan siempre las predicciones y etiquetas más recientes escritas por `09_Inference_And_Label_Enrichment` en el mismo ciclo.


## 1. Importaciones y configuración

In [0]:
exec(open("07_Utils.py").read(), globals())

In [0]:
import time
from pathlib import Path

from databricks.sdk import WorkspaceClient

In [0]:
notebook_path_raw = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook = Path(notebook_path_raw).name

# Table monitored by Databricks Lakehouse Monitoring
monitored_table = f"{catalog}.{database}.gold_fraud_inference_enriched"

# Polling interval in seconds while waiting for the refresh to complete
polling_interval_seconds = 30

print(f"Project: {project}, team: {team}, environment: {environment}")
print(f"Notebook: {notebook}")
print(f"Monitored table: {monitored_table}")


## 2. Actualización del monitor

Se lanza la actualización del monitor mediante el `SDK` de `Databricks` y se espera a que finalice consultando su estado cada `polling_interval_seconds` segundos. El ciclo termina cuando el estado es `DONE` o `FAILED`.

In [0]:
w = WorkspaceClient()

print(f"Triggering monitor refresh for: {monitored_table}")
refresh = w.quality_monitors.run_refresh(table_name = monitored_table)
refresh_id = refresh.refresh_id
print(f"Refresh identifier: {refresh_id}")
print(f"Initial state: {refresh.state.value}")
print()

while refresh.state.value in ("PENDING", "RUNNING"):
    time.sleep(polling_interval_seconds)
    refresh = w.quality_monitors.get_refresh(
        table_name = monitored_table,
        refresh_id = refresh_id
    )
    print(f"State: {refresh.state.value}")

print()
if refresh.state.value == "DONE":
    print("Monitor refresh completed successfully.")
else:
    raise RuntimeError(
        f"Monitor refresh failed with state: {refresh.state.value}. "
        f"Check the monitor logs in Catalog Explorer for details."
    )


## 3. Conclusiones y siguientes pasos

Tras la actualización, las tablas `gold_fraud_inference_enriched_profile_metrics` y `gold_fraud_inference_enriched_drift_metrics` contienen las métricas más recientes del modelo en producción. Si la alerta configurada sobre `f1_score` de la clase fraude detecta una degradación por debajo del umbral definido, el equipo recibirá una notificación para evaluar si es necesario lanzar un nuevo ciclo de reentrenamiento.